In [29]:
import phoenix as px
from qdrant_workflow import run_qdrant_rag_workflow
import os
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"

In [5]:
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
# configure the Phoenix tracer
tracer_provider = register(
  project_name="ziweidoushu", # Default is 'default'
  auto_instrument=True # Auto-instrument your app based on installed OI dependencies
)
tracer = tracer_provider.get_tracer(__name__)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: my-llm-app
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [7]:
import pandas as pd
questions = pd.read_csv("ground_truth_forLLMjudge.csv")
result = {}

for i,row in enumerate(questions.itertuples(index=False)):
    result_tmp  = run_qdrant_rag_workflow(
            birth_date=row.date,
            gender=row.gender,
            birth_hour=row.time,
            top_n_queries=5,
            question=row.question,
        )
    result[i] = {"question":result_tmp['question'],
                 "chart":result_tmp['user_chart'],
                 "queries":result_tmp['queries'],
                 "answer":result_tmp['answer']}

In [171]:
import json
from openai import OpenAI

gpt_client = OpenAI()

SYSTEM_PROMPT = """
角色：你是离线评测用的紫微斗数 RAG 评审官。
任务：仅依据“检索文段”和“命盘资料”判断助手最终回答的质量。
评分维度(数0-1,1为最佳):
1. faithfulness —— 回答是否忠于命盘，不臆造星曜、格局或结论。
2. relevance —— 回答是否紧扣用户问题，避免跑题或遗漏问点。
请在判断后给出中文说明，解释评分理由。
只输出 JSON 对象,包含键:faithfulness、relevance、explanation_faithfulness,explanation_relevance;其中 explanation 为不超过 120 字的中文说明。
""".strip()

def make_user_prompt(row):
    return f"""
用户问题：
{row['question']}

命盘资料(JSON):
{row['user_chart']}

助手最终回答：
{row['answer']}
""".strip()

In [ ]:
from phoenix.session.evaluation import get_qa_with_reference
eval_df = get_qa_with_reference(px.Client(),project_name="ziweidoushu",timeout=None)
eval_df = eval_df.reset_index()

In [172]:
def judge_example(row, model="gpt-4o"):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
    ]
    completion = gpt_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    content = completion.choices[0].message.content.strip()

    # grab the JSON object from the reply
    start = content.find("{")
    end = content.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"Judge did not return JSON: {content}")
    payload = json.loads(content[start:end + 1])
    return payload

In [173]:
results = []
for row in eval_df.itertuples(index=False):
    row_output = json.loads(row.output)
    judge_result = judge_example(row_output)
    judge_result['context_span_id'] = row[0]
    results.append(judge_result)

results = pd.DataFrame(results)
results.set_index('context_span_id')

,faithfulness,relevance,explanation_faithfulness,explanation_relevance
context_span_id,,,,
02b20ed00cfe7c5d,1.0,1.0,助手的回答忠于命盘，准确引用了官禄宫的破军、命宫的七杀和财帛宫的贪狼等星曜特性，没有臆造信息。,回答紧扣用户关于工作方向的提问，详细分析了命盘中相关星曜对职业选择的影响，提供了具体建议，未...
aacad357195727b8,0.5,0.8,助手提到的天同星和桃花星的影响部分符合命盘，但天梁星的分析与命盘不符，且未提及命盘中其他相关星曜。,回答较为全面地涉及感情发展的问题，提供了具体建议，但未完全结合命盘中所有相关信息，略有遗漏。
d1a0e02303ff6f7b,0.0,0.5,助手的回答中提到的官禄宫主星为天同和命宫中的七杀星等信息与命盘不符，存在臆造星曜的情况。,回答部分涉及事业压力和改善建议，但由于错误的星曜信息，导致部分建议不准确，未能完全紧扣用户问题。
d3072925aa8de4d7,1.0,1.0,助手的回答忠于命盘，准确引用了官禄宫、迁移宫和疾厄宫的主星及其状态，没有臆造星曜或结论。,回答紧扣用户关于换工作的疑问，详细分析了命盘中相关宫位的影响，并给出了具体建议，未跑题或遗漏问点。
40ca3ed479d6c58d,1.0,1.0,助手的回答忠于命盘，准确引用了财帛宫的七杀、禄存、咸池和天空星的影响，没有臆造星曜或结论。,回答紧扣用户关于未来财运的问题，详细分析了财帛宫的星曜影响，并提供了具体的建议，未跑题或遗漏问点。
69f6b1f9b3e4e95c,0.5,0.7,助手提到疾厄宫主星为武曲和破军，但命盘中疾厄宫无主星，破军在财帛宫。部分信息不忠于命盘。,回答涉及健康改善的可能性和建议，部分符合用户问题，但对命盘的解读有误，影响了相关性。
9ee32dfff4f79124,0.5,0.8,助手提到贪狼星在夫妻宫，但实际上贪狼星在疾厄宫。此外，夫妻宫的分析部分内容与命盘不符，影响了...,回答主要围绕家庭和婚姻问题，提供了相关建议，但部分内容与命盘不符，影响了回答的相关性。
d04d1b8ed613adb9,0.8,0.9,助手的回答大体上忠于命盘，提到了夫妻宫主星贪狼的影响和桃花星的存在，但没有提到地空星的影响，...,回答紧扣用户关于关系改善的提问，提供了具体的建议和分析，但对命盘中其他可能影响关系的因素分析...
77497c6bcf43d0d8,0.8,0.9,助手的回答大体上忠于命盘，提到了子女宫的星曜组合和大限，但未提及子女宫主星缺失的影响。,回答紧扣用户关于生育时机和子女的健康问题，但未具体说明最佳时机的年份，仅提及大限范围。


In [193]:
import numpy as np
def build_eval_df(
    result_df: pd.DataFrame,
    metric: str,
    labels: tuple[str, str] | list[str] ,
    threshold: float = 0.5) -> pd.DataFrame:

    score_col = metric
    explanation_col = f"explanation_{metric}"

    df = result_df[["context_span_id",score_col, explanation_col]].rename(
        columns={"context_span_id":"span_id",score_col: "score", explanation_col: "explanation"}
    ).copy()

    df["label"] = np.where(df["score"] > threshold, labels[1], labels[0])
    return df


In [194]:
eval_faithfulness = build_eval_df(result_df=results,metric='faithfulness',labels=['not faithful','faithful'])
eval_relevance = build_eval_df(result_df=results,metric='relevance',labels=['not relevant','relevant'])

In [195]:
from phoenix.client import Client
client = Client()


In [196]:
client.spans.log_span_annotations_dataframe(
    dataframe=eval_faithfulness,
    annotation_name="Faithfulness",
    annotator_kind="LLM",
)

client.spans.log_span_annotations_dataframe(
    dataframe=eval_relevance,
    annotation_name="Relevance",
    annotator_kind="LLM",
)
